# Model Training & Evaluation — Customer Churn Prediction

**Input:** `data/customers_features.csv` (output of feature_engineering.ipynb)

**ML Problem:** Binary Classification — predict `churn` (0 = retained, 1 = churned)

**Model:** Random Forest Classifier

### Steps
1. Load feature dataset
2. Preprocessing — encode categoricals, scale numerics
3. Train / Test split
4. Train model
5. Evaluate model
6. Feature importance
7. Generate ML output dataset

## Step 1 — Load Feature Dataset

In [28]:
import pandas as pd
import numpy as np
import json
from datetime import date

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

FEATURES_PATH  = '../../../data/customers_features.csv'
ML_OUTPUT_PATH = '../../../data/ml_output.json'

df = pd.read_csv(FEATURES_PATH)
print(f'Shape: {df.shape}')
print(f'Churn distribution:\n{df["churn"].value_counts()}')
df.head()

Shape: (5504, 13)
Churn distribution:
churn
0    4811
1     693
Name: count, dtype: int64


,customer_id,age,customer_segment,product_category,payment_type,average_order_value,discount_percentage,customer_tenure_days,return_rate,purchase_frequency,engagement_score,support_ticket_rate,churn
0,1,58.0,Premium,Sports,Cash,12622.15,6.98,894,0.111111,0.020134,3.283019,0.944444,0
1,2,20.0,Premium,Clothing,Debit Card,25515.09,1.33,949,0.866667,0.015806,1.803922,1.333333,0
2,3,55.0,Premium,Books,Credit Card,35057.90,17.01,808,0.090909,0.013614,0.700000,0.909091,0
3,4,24.0,Regular,Furniture,Net Banking,40452.85,36.49,630,0.117647,0.026984,0.116364,0.705882,1
4,5,58.0,Inactive,Clothing,Upi,3942.98,33.06,599,0.130435,0.076795,0.140940,0.152174,0


## Step 2 — Preprocessing



| Column type | Treatment | Reason |
|---|---|---|
| Categorical (`customer_segment`) | Label Encoding | Converts strings to integers |
| Numeric | StandardScaler | Zero mean, unit variance — ensures no column dominates |
| Target (`churn`) | No change | Already 0/1 integer |

In [29]:
CATEGORICAL_COLS = ['customer_segment']
NUMERIC_COLS = [
    'age', 'average_order_value', 'discount_percentage',
    'customer_tenure_days', 'return_rate', 'purchase_frequency',
    'engagement_score', 'support_ticket_rate'
]
TARGET = 'churn'

df_model = df.copy()

# Label encode categorical columns
encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    encoders[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

print('\nCategorical encoding done.')

customer_segment: {'Inactive': np.int64(0), 'New': np.int64(1), 'Premium': np.int64(2), 'Regular': np.int64(3)}

Categorical encoding done.


In [30]:
# Build feature matrix X and target vector y
FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS

X = df_model[FEATURE_COLS]
y = df_model[TARGET]

# Scale numeric columns
scaler = StandardScaler()
X = X.copy()
X[NUMERIC_COLS] = scaler.fit_transform(X[NUMERIC_COLS])

print(f'Feature matrix shape : {X.shape}')
print(f'Target vector shape  : {y.shape}')
print(f'Class balance        : {y.value_counts().to_dict()}')

Feature matrix shape : (5504, 9)
Target vector shape  : (5504,)
Class balance        : {0: 4811, 1: 693}


## Step 3 — Train / Test Split

**Split ratio:** 80% train, 20% test


In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {X_train.shape[0]} rows')
print(f'Test size  : {X_test.shape[0]} rows')
print(f'Train churn distribution: {y_train.value_counts().to_dict()}')
print(f'Test  churn distribution: {y_test.value_counts().to_dict()}')

Train size : 4403 rows
Test size  : 1101 rows
Train churn distribution: {0: 3849, 1: 554}
Test  churn distribution: {0: 962, 1: 139}


## Step 4 — Train Model

**Model chosen: Random Forest Classifier**

In [32]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print('Model training complete.')

Model training complete.


## Step 5 — Evaluate Model

In [33]:
y_pred       = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # probability of churn = 1

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_pred_proba)

print('=== Model Evaluation ===')
print(f'Accuracy  : {accuracy:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'ROC-AUC   : {roc_auc:.4f}')

=== Model Evaluation ===
Accuracy  : 0.9228
Precision : 0.6588
Recall    : 0.8058
F1 Score  : 0.7249
ROC-AUC   : 0.9599


In [34]:
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Retained (0)', 'Churned (1)']))

=== Classification Report ===
              precision    recall  f1-score   support

Retained (0)       0.97      0.94      0.96       962
 Churned (1)       0.66      0.81      0.72       139

    accuracy                           0.92      1101
   macro avg       0.81      0.87      0.84      1101
weighted avg       0.93      0.92      0.93      1101



In [35]:
print('=== Confusion Matrix ===')
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Actual: Retained', 'Actual: Churned'],
    columns=['Predicted: Retained', 'Predicted: Churned']
)
print(cm_df)

=== Confusion Matrix ===
                  Predicted: Retained  Predicted: Churned
Actual: Retained                  904                  58
Actual: Churned                    27                 112


## Step 6 — Feature Importance

Random Forest gives each feature an importance score based on how much it reduces impurity
across all trees. Higher score = stronger predictor of churn.

In [36]:
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('=== Feature Importance (sorted) ===')
print(importance_df.to_string(index=False))

=== Feature Importance (sorted) ===
             feature  importance
    engagement_score    0.540613
  purchase_frequency    0.154575
 support_ticket_rate    0.122149
customer_tenure_days    0.072107
 discount_percentage    0.028932
 average_order_value    0.026245
         return_rate    0.024195
                 age    0.023312
    customer_segment    0.007873


## Step 7 — Generate ML Output Dataset

Runs predictions on every customer and saves a structured JSON file.
This file is consumed by `POST /ai/insights` in the FastAPI app.

| Field | Source |
|---|---|
| `customer_id` | dataset |
| `churn_probability` | `model.predict_proba()` |
| `risk_level` | high > 0.7, medium 0.4–0.7, low < 0.4 |
| `top_risk_factors` | top 3 features by importance + their values |
| `customer_segment` | dataset |
| `recommended_action` | derived from risk level |
| `prediction_date` | today |
| `model_version` | v1.0 |

In [49]:
MODEL_VERSION   = 'v1.0'
PREDICTION_DATE = str(date.today())

# Predict on full dataset
X_full = df_model[FEATURE_COLS].copy()
X_full[NUMERIC_COLS] = scaler.transform(X_full[NUMERIC_COLS])
churn_probabilities = model.predict_proba(X_full)[:, 1]

print(f'Predictions generated for {len(churn_probabilities)} customers.')

Predictions generated for 5504 customers.


In [50]:
def get_risk_level(prob: float) -> str:
    if prob >= 0.7:
        return 'high'
    elif prob >= 0.4:
        return 'medium'
    return 'low'


def get_recommended_action(risk_level: str) -> str:
    actions = {
        'high':   'Immediate retention campaign - offer personalised discount or loyalty reward',
        'medium': 'Re-engagement campaign - send targeted product recommendations',
        'low':    'Maintain relationship - continue standard communication cadence'
    }
    return actions[risk_level]


def get_top_risk_factors(customer_row: pd.Series, feature_importance: pd.DataFrame, top_n: int = 3) -> list:
    top_features = feature_importance.head(top_n)['feature'].tolist()
    factors = []
    for feat in top_features:
        val = customer_row.get(feat, None)
        factors.append(f'{feat}: {round(float(val), 4) if val is not None else "N/A"}')
    return factors


print('Helper functions ready.')

Helper functions ready.


In [51]:
ml_output = []

for idx, (_, row) in enumerate(df.iterrows()):
    prob         = round(float(churn_probabilities[idx]), 4)
    risk         = get_risk_level(prob)
    risk_factors = get_top_risk_factors(row, importance_df)

    ml_output.append({
        'customer_id':        int(row['customer_id']),
        'churn_probability':  prob,
        'risk_level':         risk,
        'top_risk_factors':   risk_factors,
        'customer_segment':   str(row['customer_segment']),
        'recommended_action': get_recommended_action(risk),
        'prediction_date':    PREDICTION_DATE,
        'model_version':      MODEL_VERSION
    })

print(f'Records generated: {len(ml_output)}')
print('\nSample record:')
print(json.dumps(ml_output[0], indent=2))

Records generated: 5504

Sample record:
{
  "customer_id": 1,
  "churn_probability": 0.0212,
  "risk_level": "low",
  "top_risk_factors": [
    "engagement_score: 3.283",
    "purchase_frequency: 0.0201",
    "support_ticket_rate: 0.9444"
  ],
  "customer_segment": "Premium",
  "recommended_action": "Maintain relationship - continue standard communication cadence",
  "prediction_date": "2026-09-21",
  "model_version": "v1.0"
}


In [52]:
risk_counts = pd.Series([r['risk_level'] for r in ml_output]).value_counts()
print('Risk level distribution:')
for level, count in risk_counts.items():
    pct = count / len(ml_output) * 100
    print(f'  {level:6s}: {count:5d}  ({pct:.1f}%)')

Risk level distribution:
  low   :  4462  (81.1%)
  high  :   766  (13.9%)
  medium:   276  (5.0%)


In [53]:
with open(ML_OUTPUT_PATH, 'w') as f:
    json.dump(ml_output, f, indent=2)

print(f'Saved to : {ML_OUTPUT_PATH}')
print(f'Records  : {len(ml_output)}')

Saved to : ../../../data/ml_output.json
Records  : 5504
